# Optimizers in PyTorch

This notebook covers all 13 PyTorch optimizers and 15 learning rate schedulers through hands-on experiments.

**Sections:**
1. Setup & imports
2. Optimizer trajectory visualization on Rosenbrock function
3. Convergence speed comparison on synthetic regression
4. Adaptive methods on sparse inputs
5. L-BFGS with closure
6. Param groups & layer-wise learning rate decay
7. Gradient clipping experiment
8. Learning rate scheduler comparison
9. Super-convergence with OneCycleLR on CIFAR-10

In [ ]:
# Section 1 — Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import DataLoader, TensorDataset
import torchvision
import torchvision.transforms as transforms

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | device: {device}')

## Section 2 — Optimizer Trajectory Visualization

The **Rosenbrock function** `f(x,y) = (1−x)² + 100(y−x²)²` is the classic non-convex benchmark for optimizers.
Global minimum is at (1, 1). The valley is easy to find but hard to traverse — ideal for observing momentum effects.

In [ ]:
# Section 2 — Rosenbrock trajectories
def rosenbrock(xy):
    x, y = xy[0], xy[1]
    return (1 - x)**2 + 100 * (y - x**2)**2

def run_optimizer(opt_class, opt_kwargs, steps=500, start=(-1.5, -0.5)):
    """Run optimizer on Rosenbrock and record trajectory."""
    xy = torch.tensor(start, dtype=torch.float32, requires_grad=True)
    opt = opt_class([xy], **opt_kwargs)
    path = [xy.detach().numpy().copy()]
    for _ in range(steps):
        opt.zero_grad()
        loss = rosenbrock(xy)
        loss.backward()
        opt.step()
        path.append(xy.detach().numpy().copy())
    return np.array(path), loss.item()

optimizers_to_test = [
    ('SGD',        torch.optim.SGD,     {'lr': 0.001}),
    ('SGD+mom',    torch.optim.SGD,     {'lr': 0.001, 'momentum': 0.9}),
    ('Adam',       torch.optim.Adam,    {'lr': 0.01}),
    ('AdamW',      torch.optim.AdamW,   {'lr': 0.01}),
    ('RMSprop',    torch.optim.RMSprop, {'lr': 0.005}),
    ('Adadelta',   torch.optim.Adadelta,{'lr': 1.0}),
]

# Plot Rosenbrock surface
xs = np.linspace(-2, 2, 300)
ys = np.linspace(-1, 3, 300)
X, Y = np.meshgrid(xs, ys)
Z = (1 - X)**2 + 100 * (Y - X**2)**2

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, (name, cls, kwargs) in zip(axes.flat, optimizers_to_test):
    path, final_loss = run_optimizer(cls, kwargs)
    ax.contourf(X, Y, np.log(Z + 1), levels=40, cmap='viridis', alpha=0.7)
    ax.plot(path[:, 0], path[:, 1], 'w-', lw=1, alpha=0.6)
    ax.plot(path[0, 0], path[0, 1], 'wo', ms=5)
    ax.plot(1, 1, 'r*', ms=12, label='minimum (1,1)')
    ax.set_title(f'{name}  (final loss: {final_loss:.4f})')
    ax.legend(fontsize=8)

plt.suptitle('Optimizer Trajectories on Rosenbrock Function', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Section 3 — Convergence Speed on Synthetic Regression

In [ ]:
# Section 3 — All 13 optimizers on synthetic regression
torch.manual_seed(42)
N, D_in, D_out = 1000, 20, 1
X_data = torch.randn(N, D_in)
y_data = X_data @ torch.randn(D_in, D_out) + 0.1 * torch.randn(N, D_out)
dataset = TensorDataset(X_data, y_data)
loader  = DataLoader(dataset, batch_size=64, shuffle=True)

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(D_in, 64), nn.ReLU(),
            nn.Linear(64, 64),   nn.ReLU(),
            nn.Linear(64, D_out)
        )
    def forward(self, x): return self.net(x)

optimizer_configs = [
    ('SGD',         torch.optim.SGD,         {'lr': 0.01}),
    ('SGD+mom',     torch.optim.SGD,         {'lr': 0.01, 'momentum': 0.9}),
    ('SGD+Nesterov',torch.optim.SGD,         {'lr': 0.01, 'momentum': 0.9, 'nesterov': True}),
    ('ASGD',        torch.optim.ASGD,        {'lr': 0.01}),
    ('Rprop',       torch.optim.Rprop,       {'lr': 0.01}),
    ('Adagrad',     torch.optim.Adagrad,     {'lr': 0.01}),
    ('RMSprop',     torch.optim.RMSprop,     {'lr': 0.001}),
    ('Adadelta',    torch.optim.Adadelta,    {'lr': 1.0}),
    ('Adam',        torch.optim.Adam,        {'lr': 0.001}),
    ('AdamW',       torch.optim.AdamW,       {'lr': 0.001}),
    ('Adamax',      torch.optim.Adamax,      {'lr': 0.002}),
    ('NAdam',       torch.optim.NAdam,       {'lr': 0.002}),
    ('RAdam',       torch.optim.RAdam,       {'lr': 0.001}),
]

def train_mlp(opt_class, opt_kwargs, epochs=30):
    model = MLP()
    opt   = opt_class(model.parameters(), **opt_kwargs)
    loss_fn = nn.MSELoss()
    history = []
    for epoch in range(epochs):
        epoch_loss = 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
        history.append(epoch_loss / len(loader))
    return history

print('Training all optimizers (this may take ~60 seconds)...')
results = {}
for name, cls, kwargs in optimizer_configs:
    results[name] = train_mlp(cls, kwargs)
    print(f'  {name:20s} final loss: {results[name][-1]:.6f}')

plt.figure(figsize=(12, 6))
for name, losses in results.items():
    plt.plot(losses, label=name)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss (log scale)')
plt.yscale('log')
plt.title('Convergence Comparison — All 13 PyTorch Optimizers')
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Section 4 — Adaptive Methods on Sparse Inputs

In [ ]:
# Section 4 — Sparse input comparison (simulates bag-of-words)
torch.manual_seed(0)
VOCAB_SIZE = 5000
EMBED_DIM  = 64
N_SAMPLES  = 2000
SPARSITY   = 0.02  # 2% of vocab active per sample

# Sparse token counts
sparse_X = torch.zeros(N_SAMPLES, VOCAB_SIZE)
for i in range(N_SAMPLES):
    idx = torch.randint(0, VOCAB_SIZE, (int(VOCAB_SIZE * SPARSITY),))
    sparse_X[i, idx] = torch.randn(idx.shape[0]).abs()

W_true = torch.randn(VOCAB_SIZE, 1) * 0.1
y_sparse = sparse_X @ W_true + 0.01 * torch.randn(N_SAMPLES, 1)
sparse_loader = DataLoader(TensorDataset(sparse_X, y_sparse), batch_size=64, shuffle=True)

class LinearModel(nn.Module):
    def __init__(self): super().__init__(); self.fc = nn.Linear(VOCAB_SIZE, 1, bias=False)
    def forward(self, x): return self.fc(x)

sparse_opts = [
    ('Adagrad (sparse-friendly)', torch.optim.Adagrad,  {'lr': 0.1}),
    ('RMSprop',                   torch.optim.RMSprop,  {'lr': 0.01}),
    ('Adam',                      torch.optim.Adam,     {'lr': 0.01}),
    ('SGD',                       torch.optim.SGD,      {'lr': 0.01}),
]

sparse_results = {}
for name, cls, kwargs in sparse_opts:
    model = LinearModel()
    opt = cls(model.parameters(), **kwargs)
    losses = []
    for epoch in range(20):
        ep_loss = 0
        for xb, yb in sparse_loader:
            opt.zero_grad()
            loss = F.mse_loss(model(xb), yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item()
        losses.append(ep_loss / len(sparse_loader))
    sparse_results[name] = losses

plt.figure(figsize=(9, 5))
for name, losses in sparse_results.items():
    plt.plot(losses, label=name)
plt.xlabel('Epoch'); plt.ylabel('MSE Loss (log scale)')
plt.yscale('log')
plt.title('Adaptive Methods on Sparse Input (VOCAB_SIZE=5000, sparsity=2%)')
plt.legend()
plt.tight_layout()
plt.show()
print('\nAdagrad excels here: low-frequency features retain large effective LR.')

## Section 5 — L-BFGS with Closure

In [ ]:
# Section 5 — LBFGS closure pattern
# Fit a noisy sine curve with a small MLP
torch.manual_seed(1)
x_fit = torch.linspace(-3, 3, 200).unsqueeze(1)
y_fit = torch.sin(x_fit) + 0.1 * torch.randn_like(x_fit)

class SmallNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 32), nn.Tanh(), nn.Linear(32, 1))
    def forward(self, x): return self.net(x)

def train_lbfgs(epochs=100):
    model = SmallNet()
    optimizer = torch.optim.LBFGS(
        model.parameters(), lr=1, max_iter=20,
        history_size=100, line_search_fn='strong_wolfe'
    )
    losses = []
    for epoch in range(epochs):
        def closure():
            optimizer.zero_grad()
            pred = model(x_fit)
            loss = F.mse_loss(pred, y_fit)
            loss.backward()
            return loss
        loss = optimizer.step(closure)
        losses.append(loss.item())
    return model, losses

def train_adam_sine(epochs=500):
    model = SmallNet()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    losses = []
    for epoch in range(epochs):
        optimizer.zero_grad()
        loss = F.mse_loss(model(x_fit), y_fit)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return model, losses

model_lbfgs, losses_lbfgs = train_lbfgs(100)
model_adam,  losses_adam  = train_adam_sine(500)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.scatter(x_fit.numpy(), y_fit.numpy(), s=5, alpha=0.4, label='data')
ax1.plot(x_fit.numpy(), model_lbfgs(x_fit).detach().numpy(), 'r-', lw=2, label='L-BFGS (100 steps)')
ax1.plot(x_fit.numpy(), model_adam(x_fit).detach().numpy(), 'b--', lw=2, label='Adam (500 steps)')
ax1.set_title('Sine Curve Fit'); ax1.legend()

ax2.plot(losses_lbfgs, 'r-', label=f'L-BFGS (100 steps, final: {losses_lbfgs[-1]:.5f})')
ax2.plot(losses_adam,  'b-', label=f'Adam (500 steps, final: {losses_adam[-1]:.5f})')
ax2.set_xlabel('Steps'); ax2.set_ylabel('MSE Loss')
ax2.set_yscale('log'); ax2.set_title('Convergence: L-BFGS vs Adam'); ax2.legend()
plt.tight_layout(); plt.show()
print('L-BFGS reaches near-zero loss in far fewer steps on this small model.')

## Section 6 — Param Groups & Layer-Wise Learning Rate Decay

In [ ]:
# Section 6 — LLRD: fine-tuning with per-layer LR
# Use a 6-layer MLP to simulate a pretrained backbone + head
torch.manual_seed(42)

class DeepMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(64, 64) for _ in range(5)
        ])
        self.head = nn.Linear(64, 10)
        self.act  = nn.ReLU()
    def forward(self, x):
        for layer in self.layers:
            x = self.act(layer(x))
        return self.head(x)

X_cls = torch.randn(1000, 64)
y_cls = torch.randint(0, 10, (1000,))
cls_loader = DataLoader(TensorDataset(X_cls, y_cls), batch_size=64, shuffle=True)

def build_llrd_optimizer(model, base_lr=1e-3, decay=0.7):
    """Exponential LLRD: each deeper layer gets base_lr * decay^(depth)"""
    groups = [{'params': model.head.parameters(), 'lr': base_lr}]
    for i, layer in enumerate(reversed(model.layers)):
        lr = base_lr * (decay ** (i + 1))
        groups.append({'params': layer.parameters(), 'lr': lr})
    return torch.optim.AdamW(groups, weight_decay=0.01)

def train_cls(opt_fn, label, epochs=20):
    model = DeepMLP()
    opt = opt_fn(model)
    losses = []
    for epoch in range(epochs):
        ep_loss = 0
        for xb, yb in cls_loader:
            opt.zero_grad()
            loss = F.cross_entropy(model(xb), yb)
            loss.backward()
            opt.step()
            ep_loss += loss.item()
        losses.append(ep_loss / len(cls_loader))
    return losses

uniform_losses = train_cls(lambda m: torch.optim.AdamW(m.parameters(), lr=1e-3), 'Uniform LR')
llrd_losses    = train_cls(lambda m: build_llrd_optimizer(m, base_lr=1e-3, decay=0.7), 'LLRD (decay=0.7)')

plt.figure(figsize=(8, 5))
plt.plot(uniform_losses, label='Uniform LR (1e-3)')
plt.plot(llrd_losses,    label='LLRD (head: 1e-3, each layer × 0.7)')
plt.xlabel('Epoch'); plt.ylabel('Cross-Entropy Loss')
plt.title('Uniform LR vs Layer-Wise Learning Rate Decay (LLRD)')
plt.legend(); plt.tight_layout(); plt.show()

# Print LR assigned to each layer
model_debug = DeepMLP()
opt_debug = build_llrd_optimizer(model_debug)
print('\nLR per param group:')
for i, group in enumerate(opt_debug.param_groups):
    name = 'head' if i == 0 else f'layer[{4-i+1}]'
    print(f'  {name:12s}: lr = {group["lr"]:.6f}')

## Section 7 — Gradient Clipping Experiment

In [ ]:
# Section 7 — Gradient explosion and clipping
torch.manual_seed(0)

class DeepReLUNet(nn.Module):
    """Very deep ReLU network prone to gradient explosion with large init."""
    def __init__(self, depth=20):
        super().__init__()
        self.layers = nn.ModuleList([nn.Linear(32, 32) for _ in range(depth)])
        self.out = nn.Linear(32, 1)
        # Large init to encourage explosion
        for layer in self.layers:
            nn.init.uniform_(layer.weight, -1.5, 1.5)
    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        return self.out(x)

X_exp = torch.randn(500, 32)
y_exp = torch.randn(500, 1)
exp_loader = DataLoader(TensorDataset(X_exp, y_exp), batch_size=50, shuffle=True)

def train_with_clipping(clip_mode, max_norm=1.0, clip_val=0.5, epochs=20):
    model = DeepReLUNet(depth=20)
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    grad_norms, losses = [], []
    for epoch in range(epochs):
        ep_loss, ep_norm = 0, 0
        for xb, yb in exp_loader:
            opt.zero_grad()
            loss = F.mse_loss(model(xb), yb)
            loss.backward()
            if clip_mode == 'norm':
                norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)
            elif clip_mode == 'value':
                torch.nn.utils.clip_grad_value_(model.parameters(), clip_val)
                norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
            else:  # no clipping
                norm = sum(p.grad.norm().item() for p in model.parameters() if p.grad is not None)
            opt.step()
            ep_loss += loss.item()
            ep_norm += float(norm)
        losses.append(ep_loss / len(exp_loader))
        grad_norms.append(ep_norm / len(exp_loader))
    return losses, grad_norms

l_none,   n_none   = train_with_clipping('none')
l_norm,   n_norm   = train_with_clipping('norm',  max_norm=1.0)
l_value,  n_value  = train_with_clipping('value', clip_val=0.5)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
for losses, label in [(l_none, 'No clipping'), (l_norm, 'clip_grad_norm_(1.0)'), (l_value, 'clip_grad_value_(0.5)')]:
    ax1.plot(losses, label=label)
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()

for norms, label in [(n_none, 'No clipping'), (n_norm, 'clip_grad_norm_(1.0)'), (n_value, 'clip_grad_value_(0.5)')]:
    ax2.plot(norms, label=label)
ax2.set_title('Gradient L2 Norm'); ax2.set_xlabel('Epoch'); ax2.set_yscale('log'); ax2.legend()

plt.suptitle('Gradient Clipping: Norm vs Value vs None', fontsize=13)
plt.tight_layout(); plt.show()

## Section 8 — Learning Rate Scheduler Comparison

In [ ]:
# Section 8 — Scheduler comparison
torch.manual_seed(42)

EPOCHS = 100
STEPS_PER_EPOCH = len(loader)

def build_scheduler(name, optimizer, steps_per_epoch):
    total = EPOCHS * steps_per_epoch
    if name == 'StepLR':
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.1), 'epoch'
    elif name == 'CosineAnnealingLR':
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6), 'epoch'
    elif name == 'OneCycleLR':
        return torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=0.1, total_steps=total, pct_start=0.3), 'batch'
    elif name == 'CosineWarmRestarts':
        return torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2), 'epoch'
    elif name == 'ReduceLROnPlateau':
        return torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, min_lr=1e-7), 'plateau'

def get_lr_trace(name):
    """Simulate LR trace without training."""
    opt = torch.optim.SGD([torch.tensor(1.0)], lr=0.1)
    sched, mode = build_scheduler(name, opt, STEPS_PER_EPOCH)
    lrs = []
    for epoch in range(EPOCHS):
        if mode == 'batch':
            for _ in range(STEPS_PER_EPOCH):
                lrs.append(opt.param_groups[0]['lr'])
                sched.step()
        else:
            lrs.append(opt.param_groups[0]['lr'])
            if mode == 'epoch':
                sched.step()
            elif mode == 'plateau':
                sched.step(metrics=0.5 - epoch * 0.004)  # simulate slow improvement
    return lrs

scheduler_names = ['StepLR', 'CosineAnnealingLR', 'OneCycleLR', 'CosineWarmRestarts', 'ReduceLROnPlateau']
fig, axes = plt.subplots(1, len(scheduler_names), figsize=(18, 4))
for ax, name in zip(axes, scheduler_names):
    lrs = get_lr_trace(name)
    ax.plot(lrs)
    ax.set_title(name, fontsize=9)
    ax.set_xlabel('Step')
    ax.set_ylabel('LR')
plt.suptitle('Learning Rate Schedules Over Training', fontsize=13)
plt.tight_layout(); plt.show()

## Section 9 — Super-Convergence with OneCycleLR on CIFAR-10

**Requires GPU.** This experiment reproduces Smith's super-convergence: SGD + OneCycleLR reaches ~90% accuracy on CIFAR-10 ResNet-18 in 30 epochs versus ~100+ for StepLR.

In [ ]:
# Section 9 — OneCycleLR super-convergence on CIFAR-10
if not torch.cuda.is_available():
    print('GPU not available — skipping CIFAR-10 section. Rerun on a GPU runtime.')
else:
    transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)),
    ])
    transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.247, 0.243, 0.261)),
    ])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=transform_train)
    testset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
    trainloader = DataLoader(trainset, batch_size=128, shuffle=True,  num_workers=2)
    testloader  = DataLoader(testset,  batch_size=256, shuffle=False, num_workers=2)

    def train_cifar(scheduler_name, epochs=30):
        model = torchvision.models.resnet18(num_classes=10).to(device)
        optimizer = torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)
        if scheduler_name == 'OneCycleLR':
            scheduler = torch.optim.lr_scheduler.OneCycleLR(
                optimizer, max_lr=0.1,
                total_steps=epochs * len(trainloader),
                pct_start=0.3, anneal_strategy='cos'
            )
        else:
            scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)

        train_acc_hist, test_acc_hist = [], []
        for epoch in range(epochs):
            model.train()
            correct, total = 0, 0
            for xb, yb in trainloader:
                xb, yb = xb.to(device), yb.to(device)
                optimizer.zero_grad(set_to_none=True)
                out  = model(xb)
                loss = F.cross_entropy(out, yb)
                loss.backward()
                optimizer.step()
                if scheduler_name == 'OneCycleLR': scheduler.step()
                correct += (out.argmax(1) == yb).sum().item()
                total   += yb.size(0)
            if scheduler_name == 'StepLR': scheduler.step()
            train_acc = correct / total

            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for xb, yb in testloader:
                    xb, yb = xb.to(device), yb.to(device)
                    correct += (model(xb).argmax(1) == yb).sum().item()
                    total   += yb.size(0)
            test_acc = correct / total
            train_acc_hist.append(train_acc)
            test_acc_hist.append(test_acc)
            if (epoch + 1) % 5 == 0:
                print(f'[{scheduler_name}] Epoch {epoch+1}/{epochs}  train: {train_acc:.3f}  test: {test_acc:.3f}')
        return train_acc_hist, test_acc_hist

    one_cycle_train, one_cycle_test = train_cifar('OneCycleLR', epochs=30)
    step_lr_train,   step_lr_test   = train_cifar('StepLR',     epochs=30)

    plt.figure(figsize=(10, 5))
    plt.plot(one_cycle_test, 'g-o', ms=4, label=f'OneCycleLR (final test: {one_cycle_test[-1]:.3f})')
    plt.plot(step_lr_test,   'r-s', ms=4, label=f'StepLR     (final test: {step_lr_test[-1]:.3f})')
    plt.xlabel('Epoch'); plt.ylabel('Test Accuracy')
    plt.title('Super-Convergence: OneCycleLR vs StepLR on CIFAR-10 ResNet-18 (30 epochs)')
    plt.legend(); plt.tight_layout(); plt.show()